# 🌾 Andhra Pradesh Paddy Multi-Target Price & Spread Prediction Engine (Google Colab Notebook)
### End-to-End Production Pipeline: Data.gov.in API Ingestion, Agmarknet Market Arrival Tracking, Open-Meteo Weather Integration, Multi-Target Time-Series Modeling (Prophet + Auto-ARIMA + Order Reconciliation), Walk-Forward Backtesting & Interactive Dashboard

---
**Author:** Mandi Mitra ML Team  
**Commodity:** Paddy(Common) / Rice  
**State:** Andhra Pradesh, India  
**Architecture:** Data.gov.in API Prices ➔ GitHub Raw Arrival Tracking ➔ Weather Integration ➔ Feature Engine ➔ Multi-Target Modeling (Modal, Min, Max, Log-Spread) ➔ Ordering Reconciliation Rule ➔ Walk-Forward Backtesting ➔ 7-Day Prediction Dashboard

> [!NOTE]
> **Includes Multi-Target Price & Spread Models!** Forecasts floor price (Min), ceiling price (Max), bid-ask dispersion (Spread), and price level (Weighted Avg Modal) with strict mathematical consistency guarantees ($min \le modal \le max$).

## 🚀 How to use this notebook

1. **Run all cells** from top to bottom (`Runtime → Run all` in Colab).
2. **Wait** for data fetching (API prices + Agmarknet arrival CSV) and model training to complete (~1-2 minutes).
3. **Use the dropdown** at the bottom to select an APMC market and view its 7-day multi-target forecast.
4. **The backtest metrics** printed in Step 7 show real out-of-sample walk-forward performance.

## 📦 Step 1: Install Required Dependencies
Installs Facebook Prophet, pmdarima (Auto-ARIMA), XGBoost, scikit-learn, and ipywidgets.

In [ ]:
!pip install -q prophet pmdarima xgboost scikit-learn pandas numpy matplotlib seaborn ipywidgets

## 🔑 Step 2: Setup Configuration & API Key
Set your Data.gov.in API key below and define the GitHub Raw Arrival CSV URL.

In [ ]:
import os
import sys
import urllib.request
import urllib.parse
import json
import ssl
import time
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Enter your Data.gov.in API Key here (or use default)
API_KEY = "579b464db66ec23bdd000001a0a99e04a75a40666201931688acb738"

HISTORICAL_RESOURCE_ID = "35985678-0d79-46b4-9ed6-6f13308a1d24"
LIVE_RESOURCE_ID = "9ef84268-d588-465a-a308-a864a43d0070"

# Official Agmarknet Market Arrival CSV hosted on GitHub
ARRIVAL_CSV_URL = "https://raw.githubusercontent.com/TarunTeja44/mandiprediction/main/All_Type_of_Report_(All_Grades)_09-08-2026_07-22-58_PM.csv"

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

print("✅ Configuration & SSL context loaded successfully!")

## 🌐 Step 3: Fetch Mandi Prices (Data.gov.in API) & Market Arrival Quantities (Agmarknet CSV)
Fetches daily price records via API and merges official daily arrival volumes (Metric Tonnes) from the GitHub Agmarknet repository.

In [ ]:
AP_DISTRICTS = {
    'Alluri Sitharama Raju', 'Anakapally', 'Ananthapuramu', 'Annamayya',
    'Bapatla', 'Dr.B.R.A.Konaseema', 'East Godavari', 'Eluru',
    'Guntur', 'Kakinada', 'Krishna', 'Kurnool', 'Nandyal', 'NTR',
    'Palnadu', 'Polavaram', 'Prakasam', 'SPSR Nellore', 'Sri Potti Sriramulu Nellore',
    'Sri Sathya Sai', 'Srikakulam', 'Tirupathi', 'Vijayanagaram',
    'Visakhapatnam', 'West Godavari', 'YSR Kadapa', 'Chittoor',
    'Vizianagaram', 'Nellore', 'Kadapa'
}

def fetch_arrival_data(url=ARRIVAL_CSV_URL):
    print(f"Downloading Agmarknet daily market arrival dataset from GitHub...")
    try:
        df = pd.read_csv(url, header=1)
        arr_col = [c for c in df.columns if 'Arrival' in c and 'Quantity' in c]
        date_col = 'Date' if 'Date' in df.columns else df.columns[5]
        arr_name = arr_col[0] if arr_col else df.columns[6]
        
        df.rename(columns={date_col: 'date_str', arr_name: 'arrival_qty_mt', 'Market': 'market_raw'}, inplace=True)
        df['date'] = pd.to_datetime(df['date_str'], format='%d-%m-%Y', errors='coerce')
        df['arrival_qty_mt'] = pd.to_numeric(df['arrival_qty_mt'], errors='coerce').fillna(0.0)
        df['market'] = df['market_raw'].astype(str).str.replace(' APMC', '').str.strip()
        
        daily_arr = df.groupby(['market', 'date'])['arrival_qty_mt'].sum().reset_index()
        print(f"✓ Arrival dataset loaded: {len(daily_arr)} market-date arrival records.")
        return daily_arr
    except Exception as e:
        print(f"Arrival dataset download notice: {e}")
        return pd.DataFrame(columns=['market', 'date', 'arrival_qty_mt'])

def fetch_ap_paddy_data(api_key=API_KEY):
    print("="*75)
    print("FETCHING AP PADDY(COMMON) PRICES FROM API & MERGING AGMARKNET ARRIVALS")
    print("="*75)
    
    arr_df = fetch_arrival_data()
    all_records = []
    limit = 1000
    offset = 0
    
    print("Fetching price records from Data.gov.in API...")
    while True:
        params = {
            'api-key': api_key,
            'format': 'json',
            'limit': str(limit),
            'offset': str(offset),
            'filters[state]': 'Andhra Pradesh',
            'filters[commodity]': 'Paddy(Common)'
        }
        url = f"https://api.data.gov.in/resource/{HISTORICAL_RESOURCE_ID}?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        
        try:
            res = urllib.request.urlopen(req, context=ctx, timeout=15)
            data = json.loads(res.read().decode('utf-8'))
            records = data.get('records', [])
            if not records:
                break
            for r in records:
                dist = r.get('District', '')
                if any(d.lower() in dist.lower() for d in AP_DISTRICTS) or dist in AP_DISTRICTS:
                    all_records.append({
                        'date_str': r.get('Arrival_Date', ''),
                        'state': 'Andhra Pradesh',
                        'district': dist,
                        'market': r.get('Market', '').replace(' APMC', '').strip(),
                        'commodity': 'Paddy(Common)',
                        'min_price': r.get('Min_Price', ''),
                        'max_price': r.get('Max_Price', ''),
                        'modal_price': r.get('Modal_Price', '')
                    })
            offset += limit
            if len(records) < limit:
                break
            time.sleep(0.2)
        except Exception as e:
            print(f"  Error fetching offset {offset}: {e}")
            break
            
    print(f"  Fetched {len(all_records)} API price records.")
    
    df = pd.DataFrame(all_records)
    df['date'] = pd.to_datetime(df['date_str'], format='%d/%m/%Y', errors='coerce')
    df['modal_price'] = pd.to_numeric(df['modal_price'], errors='coerce')
    df['min_price'] = pd.to_numeric(df['min_price'], errors='coerce')
    df['max_price'] = pd.to_numeric(df['max_price'], errors='coerce')
    df = df.dropna(subset=['date', 'modal_price', 'market'])
    df = df[df['modal_price'] > 500]
    
    # Merge Arrival Quantities by (market, date)
    if not arr_df.empty:
        df = pd.merge(df, arr_df, on=['market', 'date'], how='left')
        df['arrival_qty'] = df['arrival_qty_mt'].fillna(0.0)
    else:
        df['arrival_qty'] = 0.0
        
    mkt_counts = df.groupby('market').size().sort_values(ascending=False)
    top_markets = mkt_counts[mkt_counts >= 30].index.tolist()
    
    df = df[df['market'].isin(top_markets)].sort_values(['market', 'date']).reset_index(drop=True)
    print(f"\n✓ Merged dataset ready with {len(df)} price & arrival records across {len(top_markets)} AP markets: {top_markets}")
    return df

raw_paddy_df = fetch_ap_paddy_data(API_KEY)
raw_paddy_df.head()

## ⛅ Step 4: Fetch Weather Data from Open-Meteo API
Fetches historical rainfall, temperature, and humidity data.

In [ ]:
def fetch_open_meteo_weather(start_date_str, end_date_str):
    print(f"Fetching Open-Meteo satellite weather from {start_date_str} to {end_date_str}...")
    params = {
        'latitude': '16.50',
        'longitude': '80.64',
        'start_date': start_date_str,
        'end_date': end_date_str,
        'daily': 'temperature_2m_max,temperature_2m_min,rain_sum,relative_humidity_2m_mean',
        'timezone': 'Asia/Kolkata'
    }
    url = f"https://archive-api.open-meteo.com/v1/archive?{urllib.parse.urlencode(params)}"
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        res = urllib.request.urlopen(req, timeout=20)
        w_data = json.loads(res.read().decode('utf-8'))
        daily = w_data.get('daily', {})
        w_df = pd.DataFrame({
            'date': pd.to_datetime(daily.get('time', [])),
            'temp_max': daily.get('temperature_2m_max', []),
            'temp_min': daily.get('temperature_2m_min', []),
            'rainfall': daily.get('rain_sum', []),
            'humidity': daily.get('relative_humidity_2m_mean', [])
        })
        print(f"✓ Weather dataset loaded with {len(w_df)} daily observations.")
        return w_df
    except Exception as e:
        print(f"Weather fetch warning: {e}")
        dates = pd.date_range(start_date_str, end_date_str, freq='D')
        return pd.DataFrame({
            'date': dates,
            'temp_max': 32.0, 'temp_min': 24.0, 'rainfall': 0.0, 'humidity': 75.0
        })

min_d = raw_paddy_df['date'].min().strftime('%Y-%m-%d')
max_d = datetime.date.today().strftime('%Y-%m-%d')
weather_df = fetch_open_meteo_weather(min_d, max_d)
weather_df.head()

## 🛠️ Step 5: Multi-Target Feature Engineering & Non-Trading Calendar
Computes weighted average modal price, min price, max price, price spread (`max - min`), log-spread `ln(spread + 1)`, arrival metrics (`arrival_7d_mean`, `arrival_lag_1`), and non-trading holiday calendar (`arrival_qty == 0.0`).

In [ ]:
def build_featured_dataset(df_raw, df_weather):
    print("Building multi-target feature dataset...")
    processed_markets = []
    
    for mkt, m_group in df_raw.groupby('market'):
        m_df = m_group.sort_values('date').drop_duplicates('date').reset_index(drop=True)
        dist = m_df['district'].iloc[-1] if 'district' in m_df.columns else 'Andhra Pradesh'
        
        full_dates = pd.date_range(m_df['date'].min(), m_df['date'].max(), freq='D')
        res = m_df.set_index('date').reindex(full_dates).reset_index()
        res.rename(columns={'index': 'date'}, inplace=True)
        res['market'] = mkt
        res['district'] = dist
        
        res['modal_price'] = res['modal_price'].ffill().bfill()
        res['min_price'] = res['min_price'].fillna(res['modal_price'] * 0.95).ffill().bfill()
        res['max_price'] = res['max_price'].fillna(res['modal_price'] * 1.05).ffill().bfill()
        res['arrival_qty'] = res['arrival_qty'].fillna(0.0)
        
        # Non-Trading Holiday Calendar (Requires EXACT 0.0 MT check so low-arrival trading days are learned)
        is_sunday = np.where(res['date'].dt.dayofweek == 6, 1, 0)
        no_arrivals = np.where(res['arrival_qty'] == 0.0, 1, 0)
        res['is_likely_non_trading_day'] = np.where((is_sunday == 1) | (no_arrivals == 1), 1, 0)
        
        res['weighted_avg_modal_price'] = (
            0.60 * res['modal_price'] + 0.20 * res['min_price'] + 0.20 * res['max_price']
        ).ffill().bfill()
        
        # Spread & Log-Spread (Log Transformation Enforces Positivity)
        res['spread'] = (res['max_price'] - res['min_price']).clip(lower=0.0)
        res['log_spread'] = np.log(res['spread'] + 1.0)
        
        res['arrival_lag_1'] = res['arrival_qty'].shift(1).fillna(0.0)
        res['arrival_7d_mean'] = res['arrival_qty'].shift(1).rolling(7, min_periods=1).mean().fillna(0.0)
        
        p = res['weighted_avg_modal_price']
        res['lag_1'] = p.shift(1)
        res['rolling_mean_7'] = p.shift(1).rolling(7, min_periods=1).mean()
        res['rolling_std_7'] = p.shift(1).rolling(7, min_periods=1).std().fillna(0.0)
        
        res['day_of_week'] = res['date'].dt.dayofweek
        res['month'] = res['date'].dt.month
        res['is_harvest_season'] = np.where(res['month'].isin([10, 11, 12, 4, 5]), 1, 0)
        res['msp_value'] = 2300.0
        
        res = pd.merge(res, df_weather[['date', 'rainfall', 'temp_max', 'humidity']], on='date', how='left')
        res['rainfall'] = res['rainfall'].fillna(0.0)
        res['rainfall_7d'] = res['rainfall'].shift(1).rolling(7, min_periods=1).sum().fillna(0.0)
        
        res = res.dropna(subset=['weighted_avg_modal_price', 'lag_1']).reset_index(drop=True)
        processed_markets.append(res)
        
    final_df = pd.concat(processed_markets, ignore_index=True)
    print(f"✓ Featured dataset ready: {final_df.shape[0]} rows across {final_df['market'].nunique()} markets.")
    return final_df

featured_df = build_featured_dataset(raw_paddy_df, weather_df)
featured_df[['date', 'market', 'weighted_avg_modal_price', 'min_price', 'max_price', 'spread', 'arrival_7d_mean', 'is_likely_non_trading_day']].head()

## 🤖 Step 6: Multi-Target Volatility Regime Detection & Model Training
Trains distinct models for **Weighted Avg Modal**, **Min Price**, **Max Price**, and **Log-Spread** ($z = \ln(\text{spread}+1)$) using `arrival_7d_mean` as an exogenous regressor.

In [ ]:
from prophet import Prophet
import pmdarima as pm

market_regimes = {}
prophet_modal_models = {}
prophet_min_models = {}
prophet_max_models = {}
arima_modal_models = {}
arima_min_models = {}
arima_max_models = {}
arima_spread_models = {}

print("="*80)
print("TRAINING MULTI-TARGET TIME-SERIES MODELS (MODAL, MIN, MAX, SPREAD)")
print("="*80)

for mkt, m_df in featured_df.groupby('market'):
    m_df = m_df.sort_values('date').reset_index(drop=True)
    prices = m_df['weighted_avg_modal_price']
    std_val = float(prices.std())
    
    regime = 'flat' if std_val < 5.0 else ('low_volatility' if std_val < 30.0 else 'active')
    market_regimes[mkt] = {'regime': regime, 'std': round(std_val, 2)}
    print(f"Market: {mkt:20s} | Regime: {regime:15s} | Std: Rs. {std_val:.1f}")
    
    if regime == 'flat':
        continue
        
    # 1. Prophet Models (Modal, Min, Max)
    for col, store in [('weighted_avg_modal_price', prophet_modal_models), ('min_price', prophet_min_models), ('max_price', prophet_max_models)]:
        try:
            p_df = m_df[['date', col, 'msp_value', 'rainfall_7d', 'arrival_7d_mean']].copy()
            p_df.columns = ['ds', 'y', 'msp_value', 'rainfall_7d', 'arrival_7d_mean']
            pm_m = Prophet(changepoint_prior_scale=0.1, weekly_seasonality=True, yearly_seasonality=False)
            pm_m.add_regressor('msp_value')
            pm_m.add_regressor('rainfall_7d')
            pm_m.add_regressor('arrival_7d_mean')
            pm_m.fit(p_df)
            store[mkt] = pm_m
        except Exception as e:
            pass
            
    # 2. Auto-ARIMA Models (Modal, Min, Max, Log-Spread)
    exog = m_df[['msp_value', 'rainfall_7d', 'arrival_7d_mean']].fillna(0.0).values
    for col, store in [('weighted_avg_modal_price', arima_modal_models), ('min_price', arima_min_models), ('max_price', arima_max_models)]:
        try:
            ar_m = pm.auto_arima(m_df[col].values, X=exog, seasonal=False, stepwise=True, suppress_warnings=True)
            store[mkt] = ar_m
        except Exception as e:
            pass
            
    # Spread Model (Log-Space)
    try:
        ar_sp = pm.auto_arima(m_df['log_spread'].values, X=exog, seasonal=False, stepwise=True, suppress_warnings=True)
        arima_spread_models[mkt] = ar_sp
    except Exception as e:
        pass

print("\n✓ Multi-Target Model Training Complete!")

## 🧪 Step 7: Rigorous Multi-Target Walk-Forward Backtesting
Evaluates 1-day, 3-day, and 7-day out-of-sample forecast accuracy (MAE, MAPE, RMSE, Ordering Compliance %, Range Coverage %) without data leakage.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

print("="*85)
print("RUNNING CHRONOLOGICAL WALK-FORWARD BACKTEST (NO LEAKAGE)")
print("="*85)

horizon_eval = {1: {'actual': [], 'pred': []}, 3: {'actual': [], 'pred': []}, 7: {'actual': [], 'pred': []}}
ordering_violations = 0
range_coverage_count = 0
total_test_points = 0

for mkt, m_df in featured_df.groupby('market'):
    m_df = m_df.sort_values('date').reset_index(drop=True)
    n = len(m_df)
    if n < 30:
        continue
    split_idx = int(n * 0.80)
    test_df = m_df.iloc[split_idx:].reset_index(drop=True)
    regime = market_regimes.get(mkt, {}).get('regime', 'flat')
    
    for i in range(len(test_df) - 7):
        hist = m_df.iloc[:split_idx + i]
        target_w = test_df.iloc[i:i+7]
        cur_p = float(hist['weighted_avg_modal_price'].iloc[-1])
        
        # Forecast 7 days
        if regime == 'active' and mkt in prophet_modal_models:
            f_dates = pd.date_range(pd.Timestamp(hist['date'].iloc[-1]) + pd.Timedelta(days=1), periods=7, freq='D')
            f_df = pd.DataFrame({'ds': f_dates, 'msp_value': 2300.0, 'rainfall_7d': 0.0, 'arrival_7d_mean': float(hist['arrival_7d_mean'].iloc[-1])})
            p_mod = prophet_modal_models[mkt].predict(f_df)['yhat'].values
            p_min = prophet_min_models[mkt].predict(f_df)['yhat'].values if mkt in prophet_min_models else p_mod * 0.95
            p_max = prophet_max_models[mkt].predict(f_df)['yhat'].values if mkt in prophet_max_models else p_mod * 1.05
        elif mkt in arima_modal_models:
            ex = np.tile([2300.0, 0.0, float(hist['arrival_7d_mean'].iloc[-1])], (7, 1))
            p_mod = arima_modal_models[mkt].predict(n_periods=7, X=ex)
            p_min = arima_min_models[mkt].predict(n_periods=7, X=ex) if mkt in arima_min_models else p_mod * 0.95
            p_max = arima_max_models[mkt].predict(n_periods=7, X=ex) if mkt in arima_max_models else p_mod * 1.05
        else:
            p_mod = np.full(7, cur_p)
            p_min = p_mod * 0.95
            p_max = p_mod * 1.05
            
        # Reconcile Order (min <= modal <= max)
        p_min_rec = np.minimum(p_min, p_mod - 1.0)
        p_max_rec = np.maximum(p_max, p_mod + 1.0)
        
        for h in [1, 3, 7]:
            horizon_eval[h]['actual'].append(float(target_w['weighted_avg_modal_price'].iloc[h-1]))
            horizon_eval[h]['pred'].append(float(p_mod[h-1]))
            
        for k in range(7):
            act_m = float(target_w['weighted_avg_modal_price'].iloc[k])
            if p_min_rec[k] > p_max_rec[k]:
                ordering_violations += 1
            if p_min_rec[k] <= act_m <= p_max_rec[k]:
                range_coverage_count += 1
            total_test_points += 1

print("\n--- WALK-FORWARD HORIZON METRICS MATRIX ---")
for h in [1, 3, 7]:
    act_arr = np.array(horizon_eval[h]['actual'])
    prd_arr = np.array(horizon_eval[h]['pred'])
    mae = mean_absolute_error(act_arr, prd_arr)
    mape = mean_absolute_percentage_error(act_arr, prd_arr) * 100.0
    rmse = np.sqrt(mean_squared_error(act_arr, prd_arr))
    print(f"{h}-Day Horizon ➔ MAE: Rs. {mae:>6.2f} | MAPE: {mape:>5.2f}% | RMSE: Rs. {rmse:>6.2f}")

range_cov_pct = (range_coverage_count / max(1, total_test_points)) * 100.0
ordering_viol_pct = (ordering_violations / max(1, total_test_points)) * 100.0

print(f"\nOrdering Compliance : {100.0 - ordering_viol_pct:.1f}% ({ordering_violations} violations)")
print(f"Range Coverage [Min - Max]: {range_cov_pct:.1f}% of actual modal prices inside range.")

## 🔮 Step 8: 7-Day Multi-Target Prediction & Reconciliation Engine
Generates 7-day forecasts for any selected market with ordering reconciliation ($min \le modal \le max$).

In [ ]:
def predict_7_day_forecast(market_name, forecast_days=7):
    m_df = featured_df[featured_df['market'].str.lower() == market_name.lower()].sort_values('date').reset_index(drop=True)
    if m_df.empty:
        market_name = featured_df['market'].unique()[0]
        m_df = featured_df[featured_df['market'] == market_name].sort_values('date').reset_index(drop=True)
        
    current_price = float(m_df['weighted_avg_modal_price'].iloc[-1])
    current_min = float(m_df['min_price'].iloc[-1])
    current_max = float(m_df['max_price'].iloc[-1])
    last_date = pd.to_datetime(m_df['date'].iloc[-1])
    last_arrival = float(m_df['arrival_7d_mean'].iloc[-1])
    regime = market_regimes.get(market_name, {}).get('regime', 'flat')
    
    model_used = "Prophet" if (regime == 'active' and market_name in prophet_modal_models) else ("ARIMA" if market_name in arima_modal_models else "Naive")
    future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_days, freq='D')
    
    if model_used == "Prophet":
        f_df = pd.DataFrame({'ds': future_dates, 'msp_value': 2300.0, 'rainfall_7d': 0.0, 'arrival_7d_mean': last_arrival})
        modal_raw = prophet_modal_models[market_name].predict(f_df)['yhat'].values
        min_raw = prophet_min_models[market_name].predict(f_df)['yhat'].values if market_name in prophet_min_models else modal_raw * 0.95
        max_raw = prophet_max_models[market_name].predict(f_df)['yhat'].values if market_name in prophet_max_models else modal_raw * 1.05
    elif model_used == "ARIMA":
        ex = np.tile([2300.0, 0.0, last_arrival], (forecast_days, 1))
        modal_raw = arima_modal_models[market_name].predict(n_periods=forecast_days, X=ex)
        min_raw = arima_min_models[market_name].predict(n_periods=forecast_days, X=ex) if market_name in arima_min_models else modal_raw * 0.95
        max_raw = arima_max_models[market_name].predict(n_periods=forecast_days, X=ex) if market_name in arima_max_models else modal_raw * 1.05
    else:
        modal_raw = np.full(forecast_days, current_price)
        min_raw = np.full(forecast_days, current_min)
        max_raw = np.full(forecast_days, current_max)
        
    predictions = []
    for i in range(forecast_days):
        m_val = float(modal_raw[i])
        mn_val = round(min(float(min_raw[i]), m_val - 1.0), 2)
        mx_val = round(max(float(max_raw[i]), m_val + 1.0), 2)
        sp_val = round(mx_val - mn_val, 2)
        chg = m_val - current_price
        trend = "BULLISH" if chg > 5 else ("BEARISH" if chg < -5 else "STABLE")
        
        predictions.append({
            'date': future_dates[i].strftime('%Y-%m-%d'),
            'expected_weighted_avg_price': round(m_val, 2),
            'expected_min_price': mn_val,
            'expected_max_price': mx_val,
            'expected_spread': sp_val,
            'trend': trend,
            'change': round(chg, 2)
        })
        
    return {
        'market': market_name, 'current_price': current_price, 'regime': regime, 'model_used': model_used, 'predictions': predictions
    }

sample_fc = predict_7_day_forecast(featured_df['market'].iloc[0])
print(json.dumps(sample_fc, indent=2))

## 📊 Step 9: Multi-Target Interactive Dashboard & Backtest Plotter
Select an AP Mandi Market from the dropdown widget to view Historical Modal Prices, 7-Day Forecast, Floor (Min), Ceiling (Max), and Shaded Trading Range.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

market_dropdown = widgets.Dropdown(
    options=sorted(featured_df['market'].unique()),
    value=sorted(featured_df['market'].unique())[0],
    description='AP Mandi:',
)

def render_dashboard(market):
    res = predict_7_day_forecast(market)
    m_df = featured_df[featured_df['market'] == market].sort_values('date')
    
    plt.figure(figsize=(12, 5), dpi=120)
    sns.set_theme(style="darkgrid")
    
    hist_dates = m_df['date'].tail(30)
    hist_prices = m_df['weighted_avg_modal_price'].tail(30)
    
    fc_dates = [pd.to_datetime(p['date']) for p in res['predictions']]
    fc_modals = [p['expected_weighted_avg_price'] for p in res['predictions']]
    fc_mins = [p['expected_min_price'] for p in res['predictions']]
    fc_maxs = [p['expected_max_price'] for p in res['predictions']]
    
    plot_dates = [hist_dates.iloc[-1]] + fc_dates
    plot_modals = [hist_prices.iloc[-1]] + fc_modals
    plot_mins = [hist_prices.iloc[-1]] + fc_mins
    plot_maxs = [hist_prices.iloc[-1]] + fc_maxs
    
    plt.plot(hist_dates, hist_prices, label='Historical Price', color='#10b981', linewidth=2.5, marker='o')
    plt.plot(plot_dates, plot_modals, label=f"Weighted Avg Forecast ({res['model_used']})", color='#3b82f6', linewidth=2.5, linestyle='--')
    plt.plot(plot_dates, plot_mins, label="Expected Floor (Min Price)", color='#f59e0b', linewidth=1.8, linestyle=':')
    plt.plot(plot_dates, plot_maxs, label="Expected Ceiling (Max Price)", color='#ef4444', linewidth=1.8, linestyle=':')
    plt.fill_between(plot_dates, plot_mins, plot_maxs, color='#3b82f6', alpha=0.15, label='Reconciled Trading Range [Min - Max]')
    
    plt.title(f"🌾 {market} Multi-Target Forecast — Regime: {res['regime'].upper()} ({res['model_used']} Engine)", fontsize=13, fontweight='bold')
    plt.xlabel("Date")
    plt.ylabel("Price (Rs / Quintal)")
    plt.xticks(rotation=30)
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== 📊 MULTI-TARGET 7-DAY FORECAST REPORT: {market.upper()} ===\n")
    fc_df = pd.DataFrame(res['predictions'])
    fc_df.columns = ['Date', 'Modal Price (Rs/Q)', 'Min Floor (Rs/Q)', 'Max Ceiling (Rs/Q)', 'Spread (Rs/Q)', 'Trend', 'Change (Rs)']
    display(fc_df)

widgets.interactive(render_dashboard, market=market_dropdown)